# Thành viên 2 — Kaggle `GPU T4 x 2` (32 GB VRAM): Track Quốc tế `PHEME` + `DeBERTa-v3` + Mô hình Đồ thị `SBi-FAGAT v2`

**Cấu hình bắt buộc trên Kaggle trước khi chạy song song với Thành viên 1:**
1. Bật **Accelerator: GPU T4 x 2** (tổng 32 GB VRAM).
2. Bật **Internet: ON** để tải `PHEME` từ Figshare (có kiểm tra MD5) và trọng số Hugging Face.

Bạn có 2 chế độ chạy song song:
- **`member2_en`** *(Khuyên dùng để chia 50/50 với Thành viên 1)*: Chạy trọn gói dữ liệu chuẩn quốc tế `PHEME` (`Qwen3.8-27B QLoRA` -> chưng cất 4D `DeBERTa-v3-base` -> vòng lặp tự tối ưu `SBi-FAGAT v2` -> 5-seed Inductive + Leave-One-Event-Out + Ablation).
- **`model_gnn_only`**: Chuyên trách train & tinh chỉnh **Mạng Đồ thị `SBi-FAGAT v2` + 7 mô hình so sánh** (nếu đã nạp `processed/vn` từ Thành viên 1 hoặc bật `--bootstrap-if-missing` để chạy ngay không cần chờ).

In [ ]:
# Bước 1: Kiểm tra 2 card NVIDIA Tesla T4
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| Num GPUs:", torch.cuda.device_count())

In [ ]:
# Bước 2: Clone/cập nhật repo từ GitHub & cài gói T4 x 2
%cd /kaggle/working
!if [ ! -d "gnn-nlp-propagation" ]; then git clone https://github.com/hieu-kiien/gnn-nlp-propagation.git; fi
%cd /kaggle/working/gnn-nlp-propagation
!git pull origin main
!pip install -q -U -r requirements-kaggle-t4x2.txt
!pytest -q

In [ ]:
# Bước 3: Kiểm thử khói toàn bộ đường ống (xong trong <10 giây)
!python -m scripts.kaggle_parallel_runner --role member2_en --smoke-test

In [ ]:
# Bước 4: CHẠY CHÍNH THỨC SONG SONG TRÊN KAGGLE T4 x 2
# Đổi ROLE = "model_gnn_only" và thêm --bootstrap-if-missing nếu bạn chỉ muốn chạy nhánh GNN
# Dùng bản nén sẵn 4-bit NF4 (22.4 GB) của Qwen/Qwen3.8-27B để vừa khít ổ đĩa Kaggle T4 x 2
TEACHER_MODEL = "unsloth/Qwen3.8-27B-unsloth-bnb-4bit"  # hoặc "Qwen/Qwen3.8-27B"
ROLE = "member2_en"  # hoặc "model_gnn_only"

!python -m scripts.kaggle_parallel_runner \
    --role {ROLE} \
    --lang en \
    --max-teacher-pairs 6000 \
    --teacher-model {TEACHER_MODEL} \
    --hidden-dim 256 \
    --epochs-nlp 3 \
    --epochs-gnn 100

In [ ]:
# Bước 5 (Tùy chọn cho bài báo quốc tế): Chạy thêm thực nghiệm mở rộng UPFD GossipCop
!python -m scripts.run_upfd_extension --dataset-name gossipcop --feature bert --hidden-dim 128

## Xuất kết quả gộp với Thành viên 1
Tải file `sbifagat_member2_en_bundle.zip` trong `/kaggle/working/gnn-nlp-propagation/` gồm:
- `outputs/auto_loop_pheme/loop_diagnostics.json` & `worst_val_cases_for_review.json`
- `paper/results_main_pheme.json`, `paper/results_loo_pheme.json`, `paper/results_ablation_pheme.json`
- `paper/figures/pheme/*.pdf` (đồ thị vector nét căng sẵn sàng đưa vào báo).